In [1]:
# Cell 1: Imports and Vector Store Loading
import os
from rank_bm25 import BM25Okapi
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Define paths relative to project root
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
persist_directory = os.path.join(BASE_DIR, "data", "processed", "chroma_db")

# Initialize HuggingFace Embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'}
)

# Load Chroma DB
vector_db = Chroma(
    persist_directory=persist_directory,
    embedding_function=embeddings
)
print("Chroma Vector Store loaded successfully.")

e:\enterprise-rag-assistant\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Chroma Vector Store loaded successfully.


In [2]:
# Cell 2: Test Dense Retrieval (Semantic Search)
query = "How can I fix VPN connection failure on Windows?"
top_k = 3

print(f"Testing Query: '{query}'\n")

# Dense Search Execution
dense_results = vector_db.similarity_search(query, k=top_k)

print("=== Dense Retrieval Results (ChromaDB) ===")
for idx, doc in enumerate(dense_results, 1):
    print(f"\n[Result {idx}]")
    print(doc.page_content[:200] + "...")

Testing Query: 'How can I fix VPN connection failure on Windows?'

=== Dense Retrieval Results (ChromaDB) ===

[Result 1]
you might offer to help resolve this connectivity issue. If there is a known issue or update that might rectify this problem, kindly let me know at your earliest convenience. Your prompt assistance in...

[Result 2]
request that your team prioritize investigating this issue and implementing the necessary technical solutions to restore satisfactory and stable connectivity for all affected clients. In the spirit of...

[Result 3]
able to resolve the problem internally, we need your expert support to diagnose and rectify this issue swiftly. Please consider this a high priority and provide us with the necessary technical assista...


In [3]:
# Cell 3: Test Hybrid Search Integration & Comparison
# Mocking Sparse Search dataset corpus for testing mechanics
corpus = [doc for doc in vector_db.get()['documents'][:50]] if vector_db.get()['documents'] else[]
tokenized_corpus = [doc.lower().split() for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)

# Tokenize Query for BM25
tokenized_query = query.lower().split()
doc_scores = bm25.get_scores(tokenized_query)
top_sparse_indices = sorted(range(len(doc_scores)), key=lambda i: doc_scores[i], reverse=True)[:top_k]

sparse_results = [corpus[i] for i in top_sparse_indices]

# Combine Contexts (Hybrid Pipeline Test)
combined_contexts = []

# Extraction of page_content if it's a Document object, else raw text
for doc in dense_results:
    text_content = doc.page_content if hasattr(doc, 'page_content') else str(doc)
    combined_contexts.append(text_content)

for text in sparse_results:
    if text not in combined_contexts:
        combined_contexts.append(text)

print(f"=== Hybrid Retrieval Combined Output ({len(combined_contexts)} unique chunks) ===")
for idx, text in enumerate(combined_contexts, 1):
    print(f"\n[Hybrid Chunk {idx}]")
    print(text[:200] + "...")

=== Hybrid Retrieval Combined Output (6 unique chunks) ===

[Hybrid Chunk 1]
you might offer to help resolve this connectivity issue. If there is a known issue or update that might rectify this problem, kindly let me know at your earliest convenience. Your prompt assistance in...

[Hybrid Chunk 2]
request that your team prioritize investigating this issue and implementing the necessary technical solutions to restore satisfactory and stable connectivity for all affected clients. In the spirit of...

[Hybrid Chunk 3]
able to resolve the problem internally, we need your expert support to diagnose and rectify this issue swiftly. Please consider this a high priority and provide us with the necessary technical assista...

[Hybrid Chunk 4]
Content: Dear Customer Support, I hope this message finds you well. My name is <name>, and I am reaching out regarding recurring system instability issues we have been experiencing. We are currently s...

[Hybrid Chunk 5]
Content: Dear Customer Support, I h